# Archive historique — evaluer_rappel_pipeline

Ce notebook est conservé pour documenter la préparation et les évaluations antérieures. Il ne doit pas être exécuté pour régénérer les jeux de test déjà validés.

In [ ]:
"""Évalue une sortie de pipeline contre une référence humaine validée.

Usage :
python3 Scripts/pipeline_et_validation/evaluer_rappel_pipeline.py \
  --predictions evaluation_extraction/extractions_pipeline_lot_pilote.csv
"""

from __future__ import annotations

import argparse
import csv
import re
import unicodedata
from collections import Counter
from pathlib import Path


def trouver_racine() -> Path:
    courant = Path.cwd().resolve()
    for candidat in (courant, *courant.parents):
        if (candidat / "total_donnees").is_dir() and (candidat / "evaluation_extraction").is_dir():
            return candidat
    raise FileNotFoundError("Ouvre ce notebook depuis le dossier de travail.")

RACINE = trouver_racine()
DOSSIER = RACINE / "evaluation_extraction"


def lire(chemin: Path) -> list[dict[str, str]]:
    with chemin.open(encoding="utf-8-sig", newline="") as fichier:
        return list(csv.DictReader(fichier))


def normaliser(texte: str) -> str:
    texte = "".join(
        caractere for caractere in unicodedata.normalize("NFKD", (texte or "").casefold())
        if not unicodedata.combining(caractere)
    )
    return re.sub(r"[^a-z0-9]", "", texte)


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--predictions", required=True, help="Sortie CSV de pipeline.")
    parser.add_argument("--lot", default=str(DOSSIER / "lot_pilote_12_documents.csv"))
    parser.add_argument("--reference", default=str(DOSSIER / "reference_annotations_validees.csv"))
    parser.add_argument("--rapport", default=str(DOSSIER / "RAPPORT_PIPELINE_LOT_PILOTE.md"))
    parser.add_argument("--ecarts", default=str(DOSSIER / "ecarts_pipeline_lot_pilote.csv"))
    args = parser.parse_args()

    lot = {x["document_id"] for x in lire(Path(args.lot))}
    reference = [x for x in lire(Path(args.reference)) if x["document_id"] in lot]
    predictions = [x for x in lire(Path(args.predictions)) if x["document_id"] in lot]
    niveaux = {
        "détection de personne": ("document_id", "nom"),
        "événement principal": ("document_id", "date_conseil", "nom", "fonction"),
        "événement complet": (
            "document_id", "date_conseil", "nom", "fonction", "ministere", "type_nomination",
        ),
    }

    def mesurer(champs: tuple[str, ...]) -> dict[str, object]:
        cle = lambda ligne: tuple(normaliser(ligne.get(champ, "")) for champ in champs)
        references_compte = Counter(map(cle, reference))
        predictions_compte = Counter(map(cle, predictions))
        commun = references_compte & predictions_compte
        vp = sum(commun.values())
        fp = sum((predictions_compte - references_compte).values())
        fn = sum((references_compte - predictions_compte).values())
        precision = vp / (vp + fp) if vp + fp else 0.0
        rappel = vp / (vp + fn) if vp + fn else 0.0
        f1 = 0.0 if precision + rappel == 0 else 2 * precision * rappel / (precision + rappel)
        return {
            "champs": champs,
            "references": references_compte,
            "predictions": predictions_compte,
            "vp": vp,
            "fp": fp,
            "fn": fn,
            "precision": precision,
            "rappel": rappel,
            "f1": f1,
        }

    resultats = {nom: mesurer(champs) for nom, champs in niveaux.items()}
    principal = resultats["événement principal"]

    ecarts = []
    champs_principaux = principal["champs"]
    for cle, nombre in (principal["references"] - principal["predictions"]).items():
        ecarts.extend({
            "type": "faux_negatif",
            "document_id": cle[0],
            "champs": "+".join(champs_principaux),
            "cle_normalisee": " | ".join(cle),
        } for _ in range(nombre))
    for cle, nombre in (principal["predictions"] - principal["references"]).items():
        ecarts.extend({
            "type": "faux_positif",
            "document_id": cle[0],
            "champs": "+".join(champs_principaux),
            "cle_normalisee": " | ".join(cle),
        } for _ in range(nombre))
    with Path(args.ecarts).open("w", encoding="utf-8-sig", newline="") as fichier:
        writer = csv.DictWriter(fichier, fieldnames=["type", "document_id", "champs", "cle_normalisee"])
        writer.writeheader()
        writer.writerows(ecarts)

    lignes_mesures = "\n".join(
        f"| {nom} | `{' + '.join(mesure['champs'])}` | {mesure['vp']} | {mesure['fp']} | "
        f"{mesure['fn']} | {mesure['precision']:.1%} | {mesure['rappel']:.1%} | {mesure['f1']:.1%} |"
        for nom, mesure in resultats.items()
    )

    rapport = f"""# Évaluation de la pipeline — {Path(args.lot).stem}

*Calculée à partir de la sortie indiquée à l’exécution et des {len(lot)} PDF du lot.*

La mesure normalise la casse, les accents et la ponctuation. La ligne **événement principal** est l’indicateur à retenir : une prédiction doit avoir le même document, la même date, le même nom et la même fonction que la référence. La détection de personne est conservée pour diagnostiquer les noms manqués ; elle ne suffit pas à déclarer une nomination correcte.

| Niveau | Champs exigés | VP | FP | FN | Précision | Rappel | F1 |
|---|---|---:|---:|---:|---:|---:|---:|
{lignes_mesures}

Les éventuels écarts de l’événement principal sont dans `{Path(args.ecarts).name}`. Cette évaluation sert au développement : le lot pilote ne doit pas être utilisé pour déclarer une performance finale, car ses PDF ont été consultés pendant l’amélioration des règles.
"""
    Path(args.rapport).write_text(rapport, encoding="utf-8")
    print(
        "Événement principal : "
        f"VP={principal['vp']}, FP={principal['fp']}, FN={principal['fn']}, "
        f"précision={principal['precision']:.3f}, rappel={principal['rappel']:.3f}, F1={principal['f1']:.3f}"
    )


## Exécution manuelle

Vérifier les chemins et les fichiers de sortie avant toute exécution.

In [ ]:
# main()